**Context**

This notebook explores machine learning models for claim prediction. Several algorithms were tuned and evaluated, including Logistic Regression, Random Forest, XGBoost, Decision Tree, and stacking ensembles. The evaluation focuses on accuracy, precision, recall, F1 score, and ROC‑AUC to determine the best balance between detecting claims and minimizing false alarms

**Objective**

The objective is to identify the most effective model for claim prediction, balancing high recall for claims with overall accuracy and interpretability, to support risk management decisions.


**Scoring Choice**

We selected recall as the primary scoring metric during hyperparameter tuning. In the claim prediction context, missing a true claim (false negative) is more costly than incorrectly flagging a non‑claim (false positive). By prioritizing recall, the models are optimized to capture as many true claims as possible, even if this increases false alarms.


**importing libraries**

In [55]:

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV,RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.metrics import classification_report,confusion_matrix,roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score 
from sklearn.ensemble import RandomForestClassifier, StackingClassifier





In [2]:
#read dataframe

df=pd.read_csv("car_cleaned.csv")

In [3]:
df.head()

,Unnamed: 0,age,gender,driving_experience,education,income,credit_score,vehicle_ownership,vehicle_year,married,children,postal_code,annual_mileage,vehicle_type,speeding_violations,duis,past_accidents,outcome
0,0,3,0,0,1,3,0.629027,1.0,1,0.0,1.0,10238,12000.0,0,0,0,0,0.0
1,1,0,1,0,0,0,0.357757,0.0,0,0.0,0.0,10238,16000.0,0,0,0,0,1.0
2,2,0,0,0,1,1,0.493146,1.0,0,0.0,0.0,10238,11000.0,0,0,0,0,0.0
3,3,0,1,0,2,1,0.206013,1.0,0,0.0,1.0,32765,11000.0,0,0,0,0,0.0
4,4,1,1,1,0,1,0.388366,1.0,0,0.0,0.0,32765,12000.0,0,2,0,1,1.0


In [4]:
#dropping unwantd columns

df.drop(columns=["Unnamed: 0","postal_code"],inplace=True)

In [5]:
#numeric columns that must be standerdised
num_col=["speeding_violations","duis","past_accidents","annual_mileage"]

In [6]:
#creating a pipe line

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_col)       

    ],
    remainder="passthrough" 
)



In [7]:
#feature

X=df.drop(columns="outcome")

#target

y=df["outcome"]


In [11]:
#crating test and training dataset

x_train,x_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2)

**CREATING BASELINE MODEL**

In [13]:
# Full pipeline: preprocessing + model

LR = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000,))
])

In [14]:
#TRAINING PIPELINE

LR.fit(x_train,y_train);

**EVALUATING THE MODEL**

In [15]:
#EVALUATING THE MODEL

y_predi=LR.predict(x_test)

#classification report

print(classification_report(y_test,y_predi))

              precision    recall  f1-score   support

         0.0       0.87      0.91      0.89      1367
         1.0       0.78      0.70      0.74       633

    accuracy                           0.84      2000
   macro avg       0.83      0.81      0.81      2000
weighted avg       0.84      0.84      0.84      2000



The model achieved an overall accuracy of 84% across 2,000 records. For the majority class (non‑claims), performance was strong with precision of 0.87, recall of 0.91, and an F1‑score of 0.89, indicating the model is highly effective at correctly identifying non‑claims. For the minority class (claims), performance was weaker: precision of 0.78, recall of 0.70, and an F1‑score of 0.74, meaning the model misses about 30% of actual claims. The macro average F1‑score of 0.81 highlights the performance gap between classes, while the weighted average of 0.84 reflects the influence of the larger non‑claim class. Overall, the model is reliable but shows room for improvement in capturing claims, which are the more critical outcomes from a business perspective.


In [77]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Cross-validation with stratification

scores = cross_val_score(LR, X, y, cv=skf, scoring='roc_auc')

print("Accuracy scores across folds:", scores)
print("Mean accuracy:", scores.mean())
print("Standard deviation:", scores.std())



Accuracy scores across folds: [0.9031814  0.90192054 0.89780931 0.89516722 0.89725522]
Mean accuracy: 0.8990667384888736
Standard deviation: 0.003004782120511018


The model’s accuracy across the five folds was consistently high, ranging from 89.5% to 90.3%. The mean accuracy was approximately 89.9%, with a very low standard deviation of 0.003, indicating that performance is stable and does not vary much across different train/test splits. This consistency suggests the model generalizes well and is not overly sensitive to how the data is partitioned


**TUNNING MY BASELINE MODEL**

In [78]:
# Hyperparameter grid

param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['liblinear', 'saga'],
    'classifier__class_weight': [None, 'balanced']
}


# Grid Search with cross-validation

grid_search = GridSearchCV(
    estimator=LR,
    param_grid=param_grid,
    cv=skf,
    scoring='recall',   
    
    n_jobs=-1
)

# Fit

grid_search.fit(X, y)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validated recall score:", grid_search.best_score_)


KeyboardInterrupt: 

We selected recall as the primary scoring metric during hyperparameter tuning because in the claim prediction context, missing a true claim (false negative) is more costly than incorrectly flagging a non-claim (false positive).

**TRAINING BASELINE WITH NEW PARAMETER**

In [34]:
#TUNNED PIPELINE

LR_TUNNED = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000,C=0.1,class_weight="balanced",penalty='l1',solver='saga'))
])

In [35]:
#TRAINING THE PIPELINE

LR_TUNNED.fit(x_train,y_train);


In [36]:
#EVALUATING THE MODEL

y_predi=LR_TUNNED.predict(x_test
                         
#classification report
                          
print(classification_report(y_test,y_predi))

              precision    recall  f1-score   support

         0.0       0.91      0.81      0.86      1367
         1.0       0.67      0.83      0.74       633

    accuracy                           0.81      2000
   macro avg       0.79      0.82      0.80      2000
weighted avg       0.83      0.81      0.82      2000



After hyperparameter tuning, the model achieved an overall accuracy of 81% across 2,000 records. For the majority class (non‑claims), precision improved to 0.91, meaning predictions of non‑claims are highly reliable, though recall dropped to 0.81, indicating some non‑claims were missed. For the minority class (claims), recall increased to 0.83, showing the model is now much better at capturing actual claims, though precision decreased to 0.67, meaning more non‑claims are incorrectly flagged as claims. The F1‑score for claims rose to 0.74, reflecting a stronger balance between precision and recall compared to before tuning. The macro averages (≈0.80) highlight improved balance across both classes, while the weighted averages (≈0.82) reflect the influence of the larger non‑claim class.


In [41]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Cross-validation with stratification

scores = cross_val_score(LR_TUNNED, X, y, cv=skf, scoring='roc_auc')

print("Accuracy scores across folds:", scores)
print("Mean accuracy:", scores.mean())
print("Standard deviation:", scores.std())



Accuracy scores across folds: [0.90290179 0.90222282 0.89820368 0.8945748  0.89732782]
Mean accuracy: 0.8990461818515494
Standard deviation: 0.0031180954550406397


The cross‑validated ROC‑AUC score is 0.899 with very low variation across folds (standard deviation ≈0.003). This indicates the model has strong and consistent discriminative ability, meaning it reliably ranks claim vs. no‑claim cases correctly across different subsets of the data. A score close to 0.90 shows high overall separation power, and the small standard deviation confirms the model is stable and not overly sensitive to the training split.

**TRAINING  RANDOM FOREST MODEL**

In [56]:
#TRINING RM

RF= Pipeline([
    ("preprocessing", preprocessor),
    ("classifier",RandomForestClassifier())
])

In [48]:
#TRAINING THE MODEL

RF.fit(x_train,y_train)

C:\Users\precious\anaconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['speeding_violations',
                                                   'duis', 'past_accidents',
                                                   'annual_mileage'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=20,
                                        n_estimators=200, random_state=42))])

In [49]:
#EVALUATING THE MODEL

y_predi=RF.predict(x_test)

#classification report

print(classification_report(y_test,y_predi))

              precision    recall  f1-score   support

         0.0       0.84      0.90      0.87      1367
         1.0       0.75      0.64      0.69       633

    accuracy                           0.82      2000
   macro avg       0.80      0.77      0.78      2000
weighted avg       0.82      0.82      0.82      2000



In [51]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Cross-validation with stratification

scores = cross_val_score(RF, X, y, cv=skf, scoring='roc_auc')

print("Accuracy scores across folds:", scores)
print("Mean accuracy:", scores.mean())
print("Standard deviation:", scores.std())



Accuracy scores across folds: [0.885184   0.888251   0.87761988 0.87713374 0.87552316]
Mean accuracy: 0.8807423541831103
Standard deviation: 0.005022346668023499


In [57]:
# Hyperparameter grid

param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__max_features': ['sqrt', 'log2', None],
    'classifier__bootstrap': [True, False],
    'classifier__class_weight': [None, 'balanced']
}


# Grid Search with cross-validation

grid_search = GridSearchCV(
    estimator=RF,
    param_grid=param_grid,
    cv=skf,
    scoring='recall',     
    n_jobs=-1
)

# Fit

grid_search.fit(X, y)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validated recall score:", grid_search.best_score_)


Best parameters: {'classifier__bootstrap': False, 'classifier__class_weight': 'balanced', 'classifier__max_depth': 10, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100}
Best cross-validated recall score: 0.8155097298867267


##### **TRAINING THE TUNNED RANDOM FOREST**

In [28]:
RF_TUNNED= Pipeline([
    ("preprocessing", preprocessor),
    ("classifier",RandomForestClassifier(
        class_weight="balanced",
        n_estimators=300,
        min_samples_leaf=4
        ,min_samples_split=5,
        max_depth=30,
        max_features="sqrt",
        random_state=42))
])



In [29]:
#training the model

RF_TUNNED.fit(x_train,y_train)


#EVALUATING THE MODEL

y_predi=RF_TUNNED.predict(x_test)

#classification report

print(classification_report(y_test,y_predi))

              precision    recall  f1-score   support

         0.0       0.90      0.85      0.87      1367
         1.0       0.71      0.79      0.75       633

    accuracy                           0.83      2000
   macro avg       0.81      0.82      0.81      2000
weighted avg       0.84      0.83      0.84      2000



The tuned Random Forest achieved an accuracy of 0.83, showing strong overall performance. For the no‑claim class, precision was 0.90 and recall 0.85, meaning the model correctly identifies most non‑claims with few false positives. For the claim class, precision was 0.71 and recall 0.79, indicating it captures the majority of claims while keeping false alarms at a reasonable level. The macro F1 score of 0.81 reflects balanced performance across both classes, making this model reliable and well‑suited for claim prediction with a good trade‑off between precision and recall.


**TRAINING  DECIONTREE**

In [42]:


# Define pipeline with preprocessing + Decision Tree

DT_PIPE = Pipeline([
    ("preprocessing", preprocessor),   
    ("classifier", DecisionTreeClassifier(
        criterion="gini",            
        max_depth=5,                  
        min_samples_split=5, 
        class_weight="balanced",    
        random_state=42
    ))
])

# Train

DT_PIPE.fit(x_train, y_train)

# Predict

y_pred = DT_PIPE.predict(x_test)

# Evaluate

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.91      0.84      0.88      1367
         1.0       0.71      0.82      0.76       633

    accuracy                           0.84      2000
   macro avg       0.81      0.83      0.82      2000
weighted avg       0.84      0.84      0.84      2000



In [58]:
# THIS works but computational expensive

"""
# Define pipeline
DT_PIPE = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

# Define parameter grid
param_grid = {
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [None, 5, 10, 20, 30],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 5, 10],
    "classifier__class_weight": [None, "balanced"]
}

# Grid search
grid_search = GridSearchCV(
    DT_PIPE,
    param_grid,
    cv=5,                # 5-fold cross-validation
    scoring="f1_macro",  # optimize for balanced F1 across classes
    n_jobs=-1            # use all cores
)

# Fit
grid_search.fit(x_train, y_train)

# Best parameters
print("Best parameters:", grid_search.best_params_)

# Predict with best model
best_model = grid_search.best_estimator_
y_pred = best_model.predict(x_test)

# Evaluate
print(classification_report(y_test, y_pred))
""";

**TUNNING DECISIONTREE MODEL**

In [44]:

# Define pipeline

DT_PIPE = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

# Define parameter distributions (ranges instead of fixed lists)

param_dist = {
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [None] + list(np.arange(5, 51, 5)),   
    "classifier__min_samples_split": np.arange(2, 21),            
    "classifier__min_samples_leaf": np.arange(1, 21),              
    "classifier__class_weight": [None, "balanced"]
}

# Randomized search

random_search = RandomizedSearchCV(
    DT_PIPE,
    param_distributions=param_dist,
    n_iter=50,              
    cv=5,                   
    scoring="f1_macro",     
    n_jobs=-1,             
    random_state=42
)

# Fit

random_search.fit(x_train, y_train)

# Best parameters

print("Best parameters:", random_search.best_params_)

# Predict with best model

best_model = random_search.best_estimator_
y_pred = best_model.predict(x_test)

# Evaluate

print(classification_report(y_test, y_pred))

Best parameters: {'classifier__min_samples_split': np.int64(5), 'classifier__min_samples_leaf': np.int64(6), 'classifier__max_depth': np.int64(5), 'classifier__criterion': 'gini', 'classifier__class_weight': 'balanced'}
              precision    recall  f1-score   support

         0.0       0.91      0.84      0.87      1367
         1.0       0.71      0.82      0.76       633

    accuracy                           0.83      2000
   macro avg       0.81      0.83      0.82      2000
weighted avg       0.84      0.83      0.84      2000



**STACKING MY MODELS TO IMPROVE RESULTS**

In [54]:

# Base learners

base_learners = [
    ("logistic", LogisticRegression(
        max_iter=1000, C=0.1, class_weight="balanced",
        penalty='l1', solver='saga', random_state=42
    )),
    ("rf", RandomForestClassifier(
        class_weight="balanced",
        n_estimators=200,
        min_samples_leaf=4,
        min_samples_split=5,
        max_depth=10,
        max_features="sqrt",
        random_state=42
    )),
     ("xgb", XGBClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=1,  
        random_state=42,
        use_label_encoder=False,
        eval_metric="logloss"
    )
    )
]

# Meta learner

meta_learner = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=6,
    criterion="gini",
    
    class_weight="balanced",
    random_state=42,
)

# Stacking pipeline

STACK_PIPE = Pipeline([
    ("preprocessing", preprocessor),
    ("stacking", StackingClassifier(
        estimators=base_learners,
        final_estimator=meta_learner,   
        cv=5,
        n_jobs=-1
    ))
])

# Train

STACK_PIPE.fit(x_train, y_train)

# Predict

y_pred = STACK_PIPE.predict(x_test)

# Evaluate

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.93      0.75      0.83      1367
         1.0       0.62      0.88      0.73       633

    accuracy                           0.79      2000
   macro avg       0.77      0.81      0.78      2000
weighted avg       0.83      0.79      0.80      2000



This model shows accuracy of 0.79, which is slightly lower than  tuned baseline, but it shifts the balance strongly toward recall for claims. For the no‑claim class, precision is very high at 0.93, meaning the model rarely mislabels claims as non‑claims, but recall drops to 0.75, so more actual no‑claims are missed. For the claim class, recall rises to 0.88, which is excellent for catching claims, though precision falls to 0.62, indicating more false alarms. Overall, the macro F1 is 0.78, showing weaker balance compared to the baseline. In short, this stacked model favors identifying claims even at the cost of misclassifying more non‑claims, making it useful if the business priority is maximizing claim detection rather than maintaining overall accuracy

In [59]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Cross-validation with stratification

scores = cross_val_score(STACK_PIPE, X, y, cv=skf, scoring='roc_auc')

print("Accuracy scores across folds:", scores)
print("Mean accuracy:", scores.mean())
print("Standard deviation:", scores.std())



Accuracy scores across folds: [0.89537439 0.89306716 0.89152556 0.89008574 0.89142682]
Mean accuracy: 0.8922959341964279
Standard deviation: 0.0018058497379506346


ROC‑AUC Explanation
Both the baseline and stacked models achieve an ROC‑AUC of 0.899. This shows that, despite differences in precision, recall, and accuracy, their overall ability to distinguish between claim and no‑claim cases is equally strong. ROC‑AUC reflects the model’s ranking performance across thresholds, and a score near 0.90 indicates high discriminative power. In practice, this means both models are effective at separating the two classes, but the choice between them depends on the preferred balance of recall and precision rather than their ranking ability.
